In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

from utility import data_preparation,document_cleaning

import numpy as np

/home/michele/.local/lib/python3.10/site-packages/beir/datasets/data_loader.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [4]:
corpus, keys = data_preparation("scifact")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [7]:
cleaned_corpus=document_cleaning(corpus)

documents cleaning:   0%|          | 0/5183 [00:00<?, ?it/s]

In [8]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(cleaned_corpus)

In [9]:
len(vectorizer.vocabulary_)

31637

In [10]:
X

<5183x31637 sparse matrix of type '<class 'numpy.float64'>'
	with 445270 stored elements in Compressed Sparse Row format>

In [11]:
scores = X.dot( X.transpose() )

scores

<5183x5183 sparse matrix of type '<class 'numpy.float64'>'
	with 25969655 stored elements in Compressed Sparse Row format>

In [12]:
def documents_pairs(scores: np.array, threshold: float):
    
    index__doc_id_map=dict(zip(range(0,len(corpus)),keys))
    
    np.fill_diagonal(scores, 0)
    
    pairs = np.argwhere(scores >= threshold)
    
    unique_pairs=set(tuple(sorted(p)) for p in pairs)
    
    return [((index__doc_id_map[index1],index__doc_id_map[index2]),scores[index1,index2]) for index1, index2 in unique_pairs]

In [26]:
pairs = documents_pairs(scores.toarray(), 0.45)

In [27]:
import pickle
with open('spark_list.pkl', 'rb') as f:
    spark_list = sorted(pickle.load(f))

seq_list=sorted(pairs)


print(set(spark_list)-set(seq_list))

print(set(seq_list)-set(spark_list))

set()
{(('1684489', '2991954'), 0.4949900189998041)}
